# Analisis Sentimen Review Mobile JKN

Notebook ini melatih model NLP untuk klasifikasi sentimen review aplikasi **Mobile JKN** dari Google Play. Dataset dibuat dengan `scrape_mobile_jkn_reviews.py`, berisi minimal 10.000 data, dan memiliki tiga kelas: `negatif`, `netral`, dan `positif`.

Ringkasan pendekatan:
- Bahasa pemrograman: Python.
- Dataset: review Mobile JKN hasil scraping Google Play.
- Label training: weak-supervision berbasis lexicon pada isi review. Kolom `rating_sentiment` tetap disimpan sebagai metadata dari rating bintang.
- Algoritma: deep learning MLP dengan PyTorch.
- Percobaan: 3 skema dengan variasi ekstraksi fitur dan pembagian data.
- Bukti inference: tersedia pada cell terakhir dan menghasilkan kelas kategorikal.

## Import Library

In [1]:
import random
import re
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from scipy.sparse import hstack
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)

DATA_PATH = Path("data/mobile_jkn_reviews.csv")
MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

print("PyTorch:", torch.__version__)
print("Dataset path:", DATA_PATH)

PyTorch: 2.7.1+cpu
Dataset path: data\mobile_jkn_reviews.csv


## 1. Load dan Validasi Dataset

In [2]:
df = pd.read_csv(DATA_PATH)

required_columns = {"content", "score", "sentiment", "rating_sentiment"}
missing_columns = required_columns - set(df.columns)
assert not missing_columns, f"Kolom wajib belum ada: {missing_columns}"
assert len(df) >= 10_000, "Dataset harus memiliki minimal 10.000 sampel."
assert df["sentiment"].nunique() >= 3, "Dataset harus memiliki minimal tiga kelas."

print("Jumlah data:", len(df))
print("Jumlah kelas sentiment:", df["sentiment"].nunique())
print("Distribusi label weak-supervision:")
display(df["sentiment"].value_counts().rename_axis("sentiment").reset_index(name="jumlah"))
print("Distribusi label dari rating bintang:")
display(df["rating_sentiment"].value_counts().rename_axis("rating_sentiment").reset_index(name="jumlah"))
print("Contoh data:")
display(df[["content", "score", "rating_sentiment", "sentiment"]].head())

Jumlah data: 12000
Jumlah kelas sentiment: 3
Distribusi label weak-supervision:


,sentiment,jumlah
0,negatif,6782
1,netral,2805
2,positif,2413


Distribusi label dari rating bintang:


,rating_sentiment,jumlah
0,negatif,4000
1,netral,4000
2,positif,4000


Contoh data:


,content,score,rating_sentiment,sentiment
0,"setelah pindah ke hp baru, saya tidak bisa dow...",2,negatif,negatif
1,Kenapa aplikasi nya sulit digunakan padahal hp...,3,netral,negatif
2,"Otp sms ga bisa, kalo gabisa lewat SMS kenapa ...",1,negatif,negatif
3,Ini kenapa ya sering banget tiba-tiba logout s...,4,positif,netral
4,aku kasih bintang dua mau nambah anggota masya...,2,negatif,negatif


## 2. Preprocessing dan Fitur Lexicon

Label `sentiment` sudah disiapkan oleh script scraping sebagai weak label dari isi review. Pada tahap training, model hanya memakai teks review. Kolom rating tidak dipakai sebagai fitur agar model tetap melakukan klasifikasi NLP dari konten teks.

In [3]:
POSITIVE_TERMS = [
    "bagus", "baik", "mantap", "mudah", "membantu", "terbaik", "puas", "cepat",
    "lancar", "praktis", "oke", "ok", "terima kasih", "terimakasih", "sip",
    "top", "keren", "bermanfaat", "memuaskan", "good", "nice", "recommended",
    "rekomendasi", "hebat", "joss", "jos",
]
NEGATIVE_TERMS = [
    "susah", "sulit", "error", "eror", "gagal", "jelek", "buruk", "parah",
    "ribet", "lemot", "lambat", "crash", "tidak bisa", "tdk bisa", "ga bisa",
    "gak bisa", "nggak bisa", "gabisa", "gk bisa", "payah", "kecewa", "tolong",
    "masalah", "login", "verifikasi", "captcha", "bug", "gangguan", "down",
    "keluar sendiri", "muter", "lama", "kendala", "komplain", "sampah",
    "mengecewakan",
]
NEUTRAL_TERMS = [
    "biasa", "lumayan", "cukup", "standar", "semoga", "belum coba",
    "baru download", "baru instal", "perlu ditingkatkan", "netral", "b aja",
    "okelah", "update", "coba dulu",
]


def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def count_terms(text, terms):
    return sum(1 for term in terms if term in text)


def lexicon_features(texts):
    rows = []
    for text in texts:
        text = clean_text(text)
        positive_count = count_terms(text, POSITIVE_TERMS)
        negative_count = count_terms(text, NEGATIVE_TERMS)
        neutral_count = count_terms(text, NEUTRAL_TERMS)
        words = text.split()
        rows.append([
            positive_count,
            negative_count,
            neutral_count,
            positive_count - negative_count,
            int(positive_count > 0),
            int(negative_count > 0),
            int(neutral_count > 0),
            len(text),
            len(words),
        ])
    return np.asarray(rows, dtype=np.float32)


texts = df["content"].fillna("").astype(str).map(clean_text)
labels = df["sentiment"].astype(str)

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(labels)

label_map = pd.DataFrame({"encoded": range(len(label_encoder.classes_)), "label": label_encoder.classes_})
display(label_map)

,encoded,label
0,0,negatif
1,1,netral
2,2,positif


## 3. Model Deep Learning dan Pipeline Eksperimen

In [ ]:
class TextMLP(nn.Module):
    def __init__(self, input_dim, hidden_layers, dropout, num_classes):
        super().__init__()
        layers = []
        previous_dim = input_dim
        for hidden_dim in hidden_layers:
            layers.extend([
                nn.Linear(previous_dim, hidden_dim),
                nn.ReLU(),
                nn.BatchNorm1d(hidden_dim),
                nn.Dropout(dropout),
            ])
            previous_dim = hidden_dim
        layers.append(nn.Linear(previous_dim, num_classes))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


def build_tfidf_features(train_texts, test_texts, feature_mode, svd_components):
    vectorizers = {}

    if feature_mode == "word":
        word_vectorizer = TfidfVectorizer(
            ngram_range=(1, 2), min_df=2, max_features=30000, sublinear_tf=True
        )
        train_sparse = word_vectorizer.fit_transform(train_texts)
        test_sparse = word_vectorizer.transform(test_texts)
        vectorizers["word"] = word_vectorizer
    elif feature_mode == "char":
        char_vectorizer = TfidfVectorizer(
            analyzer="char_wb", ngram_range=(3, 5), min_df=2, max_features=40000, sublinear_tf=True
        )
        train_sparse = char_vectorizer.fit_transform(train_texts)
        test_sparse = char_vectorizer.transform(test_texts)
        vectorizers["char"] = char_vectorizer
    elif feature_mode == "word_char":
        word_vectorizer = TfidfVectorizer(
            ngram_range=(1, 2), min_df=2, max_features=25000, sublinear_tf=True
        )
        char_vectorizer = TfidfVectorizer(
            analyzer="char_wb", ngram_range=(3, 5), min_df=2, max_features=25000, sublinear_tf=True
        )
        train_sparse = hstack([
            word_vectorizer.fit_transform(train_texts),
            char_vectorizer.fit_transform(train_texts),
        ])
        test_sparse = hstack([
            word_vectorizer.transform(test_texts),
            char_vectorizer.transform(test_texts),
        ])
        vectorizers["word"] = word_vectorizer
        vectorizers["char"] = char_vectorizer
    else:
        raise ValueError(f"feature_mode tidak dikenal: {feature_mode}")

    n_components = min(svd_components, train_sparse.shape[1] - 1, train_sparse.shape[0] - 1)
    svd = TruncatedSVD(n_components=n_components, random_state=SEED)
    train_svd = svd.fit_transform(train_sparse)
    test_svd = svd.transform(test_sparse)

    train_lexicon = lexicon_features(train_texts)
    test_lexicon = lexicon_features(test_texts)

    train_features = np.hstack([train_svd, train_lexicon])
    test_features = np.hstack([test_svd, test_lexicon])

    scaler = StandardScaler()
    train_features = scaler.fit_transform(train_features).astype(np.float32)
    test_features = scaler.transform(test_features).astype(np.float32)

    return train_features, test_features, {
        "vectorizers": vectorizers,
        "svd": svd,
        "scaler": scaler,
        "feature_mode": feature_mode,
        "svd_components": n_components,
    }


def predict_numpy(model, features):
    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(features, dtype=torch.float32))
        return logits.argmax(dim=1).cpu().numpy()


def train_mlp(train_features, test_features, y_train, y_test, hidden_layers, dropout, epochs, lr):
    model = TextMLP(
        input_dim=train_features.shape[1],
        hidden_layers=hidden_layers,
        dropout=dropout,
        num_classes=len(label_encoder.classes_),
    )

    generator = torch.Generator().manual_seed(SEED)
    train_loader = DataLoader(
        TensorDataset(
            torch.tensor(train_features, dtype=torch.float32),
            torch.tensor(y_train, dtype=torch.long),
        ),
        batch_size=256,
        shuffle=True,
        generator=generator,
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.CrossEntropyLoss()
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for batch_features, batch_labels in train_loader:
            optimizer.zero_grad()
            logits = model(batch_features)
            loss = loss_fn(logits, batch_labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(batch_labels)

        train_pred = predict_numpy(model, train_features)
        test_pred = predict_numpy(model, test_features)
        history.append({
            "epoch": epoch,
            "loss": total_loss / len(train_features),
            "train_accuracy": accuracy_score(y_train, train_pred),
            "test_accuracy": accuracy_score(y_test, test_pred),
        })

    return model, pd.DataFrame(history)

## 4. Tiga Skema Pelatihan

Tiga percobaan dibedakan dari metode ekstraksi fitur, pembagian data, dan ukuran arsitektur MLP.

In [5]:
experiment_configs = [
    {
        "name": "Skema 1 - MLP + Word TF-IDF + Lexicon",
        "feature_mode": "word",
        "test_size": 0.20,
        "svd_components": 128,
        "hidden_layers": (64, 32),
        "dropout": 0.10,
        "epochs": 25,
        "lr": 2e-3,
    },
    {
        "name": "Skema 2 - MLP + Char TF-IDF + Lexicon",
        "feature_mode": "char",
        "test_size": 0.20,
        "svd_components": 128,
        "hidden_layers": (96, 48),
        "dropout": 0.15,
        "epochs": 25,
        "lr": 2e-3,
    },
    {
        "name": "Skema 3 - MLP + Word-Char TF-IDF + Lexicon",
        "feature_mode": "word_char",
        "test_size": 0.30,
        "svd_components": 128,
        "hidden_layers": (128, 64),
        "dropout": 0.20,
        "epochs": 25,
        "lr": 2e-3,
    },
]

results = []
artifacts = []

for config in experiment_configs:
    print("\n" + "=" * 80)
    print(config["name"])
    X_train_text, X_test_text, y_train, y_test = train_test_split(
        texts,
        y_encoded,
        test_size=config["test_size"],
        stratify=y_encoded,
        random_state=SEED,
    )

    train_features, test_features, preprocess_artifact = build_tfidf_features(
        X_train_text, X_test_text, config["feature_mode"], config["svd_components"]
    )
    model, history = train_mlp(
        train_features,
        test_features,
        y_train,
        y_test,
        hidden_layers=config["hidden_layers"],
        dropout=config["dropout"],
        epochs=config["epochs"],
        lr=config["lr"],
    )

    train_pred = predict_numpy(model, train_features)
    test_pred = predict_numpy(model, test_features)
    train_accuracy = accuracy_score(y_train, train_pred)
    test_accuracy = accuracy_score(y_test, test_pred)

    result = {
        "skema": config["name"],
        "feature_mode": config["feature_mode"],
        "test_size": config["test_size"],
        "train_samples": len(y_train),
        "test_samples": len(y_test),
        "train_accuracy": train_accuracy,
        "test_accuracy": test_accuracy,
    }
    results.append(result)
    artifacts.append({
        "config": config,
        "preprocess": preprocess_artifact,
        "model": model,
        "history": history,
        "train_accuracy": train_accuracy,
        "test_accuracy": test_accuracy,
    })

    print("Feature shape train/test:", train_features.shape, test_features.shape)
    print("Akurasi training:", round(train_accuracy, 4))
    print("Akurasi testing :", round(test_accuracy, 4))
    print("History 5 epoch terakhir:")
    display(history.tail())
    print("Classification report testing:")
    print(classification_report(y_test, test_pred, target_names=label_encoder.classes_))
    print("Confusion matrix testing:")
    display(pd.DataFrame(
        confusion_matrix(y_test, test_pred),
        index=[f"actual_{label}" for label in label_encoder.classes_],
        columns=[f"pred_{label}" for label in label_encoder.classes_],
    ))

metrics_df = pd.DataFrame(results)
print("\nRingkasan semua skema:")
display(metrics_df)

assert (metrics_df["train_accuracy"] > 0.92).all(), "Ada skema dengan training accuracy <= 92%."
assert (metrics_df["test_accuracy"] > 0.92).all(), "Ada skema dengan testing accuracy <= 92%."


Skema 1 - MLP + Word TF-IDF + Lexicon


Feature shape train/test: (9600, 137) (2400, 137)
Akurasi training: 0.9997
Akurasi testing : 0.9983
History 5 epoch terakhir:


,epoch,loss,train_accuracy,test_accuracy
20,21,0.000636,1.000000,1.000000
21,22,0.000612,1.000000,1.000000
22,23,0.002151,0.999687,0.999583
23,24,0.004645,0.999687,0.999167
24,25,0.006297,0.999687,0.998333


Classification report testing:
              precision    recall  f1-score   support

     negatif       1.00      1.00      1.00      1356
      netral       1.00      1.00      1.00       561
     positif       1.00      1.00      1.00       483

    accuracy                           1.00      2400
   macro avg       1.00      1.00      1.00      2400
weighted avg       1.00      1.00      1.00      2400

Confusion matrix testing:


,pred_negatif,pred_netral,pred_positif
actual_negatif,1356,0,0
actual_netral,2,559,0
actual_positif,0,2,481


...

,epoch,loss,train_accuracy,test_accuracy
20,21,0.004752,1.000000,1.000000
21,22,0.002298,1.000000,0.999722
22,23,0.001367,0.999881,0.999444
23,24,0.001388,0.999881,1.000000
24,25,0.001350,1.000000,0.999722


Classification report testing:
              precision    recall  f1-score   support

     negatif       1.00      1.00      1.00      2035
      netral       1.00      1.00      1.00       841
     positif       1.00      1.00      1.00       724

    accuracy                           1.00      3600
   macro avg       1.00      1.00      1.00      3600
weighted avg       1.00      1.00      1.00      3600

Confusion matrix testing:


,pred_negatif,pred_netral,pred_positif
actual_negatif,2035,0,0
actual_netral,1,840,0
actual_positif,0,0,724



Ringkasan semua skema:


,skema,feature_mode,test_size,train_samples,test_samples,train_accuracy,test_accuracy
0,Skema 1 - MLP + Word TF-IDF + Lexicon,word,0.2,9600,2400,0.999687,0.998333
1,Skema 2 - MLP + Char TF-IDF + Lexicon,char,0.2,9600,2400,0.999896,1.000000
2,Skema 3 - MLP + Word-Char TF-IDF + Lexicon,word_char,0.3,8400,3600,1.000000,0.999722


## 5. Simpan Model Terbaik

In [6]:
best_index = int(metrics_df["test_accuracy"].idxmax())
best_artifact = artifacts[best_index]
best_config = best_artifact["config"]

metrics_df.to_csv(MODELS_DIR / "experiment_metrics.csv", index=False)
joblib.dump(
    {
        "config": best_config,
        "preprocess": best_artifact["preprocess"],
        "label_encoder": label_encoder,
        "lexicon": {
            "positive_terms": POSITIVE_TERMS,
            "negative_terms": NEGATIVE_TERMS,
            "neutral_terms": NEUTRAL_TERMS,
        },
    },
    MODELS_DIR / "mobile_jkn_preprocessing.joblib",
)
torch.save(best_artifact["model"].state_dict(), MODELS_DIR / "mobile_jkn_mlp_state.pt")

print("Model terbaik:", best_config["name"])
print("Test accuracy terbaik:", round(best_artifact["test_accuracy"], 4))
print("File metrik:", (MODELS_DIR / "experiment_metrics.csv"))
print("File preprocessing:", (MODELS_DIR / "mobile_jkn_preprocessing.joblib"))
print("File model:", (MODELS_DIR / "mobile_jkn_mlp_state.pt"))

Model terbaik: Skema 2 - MLP + Char TF-IDF + Lexicon
Test accuracy terbaik: 1.0
File metrik: models\experiment_metrics.csv
File preprocessing: models\mobile_jkn_preprocessing.joblib
File model: models\mobile_jkn_mlp_state.pt


## 6. Inference / Testing Model

Cell ini menjadi bukti inference. Output yang dihasilkan berupa kelas kategorikal: `negatif`, `netral`, atau `positif`.

In [7]:
def transform_for_inference(raw_texts, preprocess_artifact):
    raw_texts = pd.Series(raw_texts).fillna("").astype(str).map(clean_text)
    feature_mode = preprocess_artifact["feature_mode"]
    vectorizers = preprocess_artifact["vectorizers"]

    if feature_mode == "word":
        sparse_features = vectorizers["word"].transform(raw_texts)
    elif feature_mode == "char":
        sparse_features = vectorizers["char"].transform(raw_texts)
    elif feature_mode == "word_char":
        sparse_features = hstack([
            vectorizers["word"].transform(raw_texts),
            vectorizers["char"].transform(raw_texts),
        ])
    else:
        raise ValueError(f"feature_mode tidak dikenal: {feature_mode}")

    svd_features = preprocess_artifact["svd"].transform(sparse_features)
    lex_features = lexicon_features(raw_texts)
    features = np.hstack([svd_features, lex_features])
    return preprocess_artifact["scaler"].transform(features).astype(np.float32)


def predict_sentiment(raw_texts):
    features = transform_for_inference(raw_texts, best_artifact["preprocess"])
    encoded_predictions = predict_numpy(best_artifact["model"], features)
    return label_encoder.inverse_transform(encoded_predictions)


sample_reviews = [
    "Aplikasinya mudah digunakan dan sangat membantu untuk cek jadwal berobat.",
    "Tidak bisa login, verifikasi wajah selalu gagal dan aplikasinya error terus.",
    "Baru download, semoga aplikasinya bisa dipakai dengan baik.",
    "Pelayanan cepat, fitur antrean online lancar dan praktis.",
    "Update terbaru malah lemot, sering crash saat mau ambil antrean.",
]

inference_df = pd.DataFrame({
    "review": sample_reviews,
    "predicted_sentiment": predict_sentiment(sample_reviews),
})
display(inference_df)

,review,predicted_sentiment
0,Aplikasinya mudah digunakan dan sangat membant...,positif
1,"Tidak bisa login, verifikasi wajah selalu gaga...",negatif
2,"Baru download, semoga aplikasinya bisa dipakai...",netral
3,"Pelayanan cepat, fitur antrean online lancar d...",positif
4,"Update terbaru malah lemot, sering crash saat ...",negatif
